# Inferencia Standalone — `be_payment_reminder` — adaptado a SeqNN **v1.9.0**

Notebook de scoring productivo de kernel limpio. Reconstruye el modelo y la
calibración desde los artefactos persistidos por el notebook de entrenamiento
`seqnn_recurrent_charge_gone` (v1.9.0) **sin re-entrenar**.

**Cambios vs. la versión v1.8.x de este notebook**

1. Apunta a `artifacts_v1_9_0/` y a los modelos `*_model_v1_9_0.pt`.
2. Todo (modelo, decoder, temperaturas, features, política de abstención) se
   lee de `inference_bundle_v1_9_0.json` — cero valores hardcodeados.
3. `HMM_FEATURES` se toma íntegro de `feature_manifest.json` (47 features). Se
   elimina el hack `BASELINE[:input_size] + NEW_FEATURE_NAMES`.
4. Se incluyen las 5 arquitecturas + `create_model`, así que carga cualquier
   modelo seleccionado (RNN/GRU/LSTM/TCN/DEEP_GRU) **o el ENSEMBLE**.
5. Feature engineering renombrado a la convención genérica `target_*`
   (el SQL v5.0.1 de Google One ya expone `target_txn_*`).
6. Los cortes de probabilidad acumulada se derivan de `bucket_edges`.
7. Política de abstención del clasificador de censura cableada como toggle.


## 1. Imports + arquitecturas (idénticas al training v1.9.0)

In [5]:
import polars as pl
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import gc, time, json, joblib, pickle
from pathlib import Path
from datetime import datetime, timedelta, timezone
from google.cloud import bigquery

# ═══════════════════════════════════════════════════════════════════════════
#  ARQUITECTURAS — copiadas verbatim de la celda 28 del training v1.9.0.
#  Las definiciones deben coincidir 1:1 con las del training para que
#  load_state_dict() no falle (mismas keys, mismos shapes).
# ═══════════════════════════════════════════════════════════════════════════

class TemporalAttention(nn.Module):
    """Atención aprendida sobre T timesteps."""
    def __init__(self, hidden_size: int):
        super().__init__()
        self.score = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, out, lengths=None):
        scores = self.score(out).squeeze(-1)
        if lengths is not None:
            T = out.size(1)
            mask = torch.arange(T, device=out.device).unsqueeze(0) < lengths.unsqueeze(1)
            scores = scores.masked_fill(~mask, float('-inf'))
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        return (out * weights).sum(dim=1)


class MultitaskHeadsMixin:
    """Cabezas auxiliares multitask + regression head."""
    def _build_multitask_heads(self, hidden_size):
        self.fc_y3d  = nn.Linear(hidden_size, 1)
        self.fc_y7d  = nn.Linear(hidden_size, 1)
        self.fc_y14d = nn.Linear(hidden_size, 1)
        self.fc_y30d = nn.Linear(hidden_size, 1)

    def _multitask_forward(self, pooled):
        return torch.cat([self.fc_y3d(pooled), self.fc_y7d(pooled),
                          self.fc_y14d(pooled), self.fc_y30d(pooled)], dim=1)

    def _shared_head_forward(self, pooled):
        pooled = self.norm(pooled)
        pooled = self.dropout(pooled)
        logits_bucket = self.fc_bucket(pooled)
        pred_days = F.relu(self.fc_days(pooled)).squeeze(-1)
        logits_mt = self._multitask_forward(pooled) if self.use_multitask else None
        return logits_bucket, logits_mt, pred_days


class GRUModel(MultitaskHeadsMixin, nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, n_buckets,
                 dropout=0.3, use_multitask=True):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True,
                          dropout=dropout if num_layers > 1 else 0.0)
        self.attn = TemporalAttention(hidden_size)
        self.norm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.fc_bucket = nn.Linear(hidden_size, n_buckets)
        self.fc_days = nn.Linear(hidden_size, 1)
        self.use_multitask = use_multitask
        if use_multitask:
            self._build_multitask_heads(hidden_size)

    def forward(self, x, lengths=None):
        out, _ = self.gru(x)
        pooled = self.attn(out, lengths)
        return self._shared_head_forward(pooled)


class RNNModel(MultitaskHeadsMixin, nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, n_buckets,
                 dropout=0.3, use_multitask=True):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True,
                          nonlinearity='tanh', dropout=dropout if num_layers > 1 else 0.0)
        self.attn = TemporalAttention(hidden_size)
        self.norm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.fc_bucket = nn.Linear(hidden_size, n_buckets)
        self.fc_days = nn.Linear(hidden_size, 1)
        self.use_multitask = use_multitask
        if use_multitask:
            self._build_multitask_heads(hidden_size)

    def forward(self, x, lengths=None):
        out, _ = self.rnn(x)
        pooled = self.attn(out, lengths)
        return self._shared_head_forward(pooled)


class LSTMModel(MultitaskHeadsMixin, nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, n_buckets,
                 dropout=0.3, use_multitask=True):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.attn = TemporalAttention(hidden_size)
        self.norm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.fc_bucket = nn.Linear(hidden_size, n_buckets)
        self.fc_days = nn.Linear(hidden_size, 1)
        self.use_multitask = use_multitask
        if use_multitask:
            self._build_multitask_heads(hidden_size)

    def forward(self, x, lengths=None):
        out, _ = self.lstm(x)
        pooled = self.attn(out, lengths)
        return self._shared_head_forward(pooled)


class CausalConvBlock(nn.Module):
    """Bloque causal dilatado con residual. Padding solo a la izquierda."""
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout=0.3):
        super().__init__()
        self.causal_pad = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size,
                              dilation=dilation, padding=0, bias=True)
        self.bn = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.res_proj = (nn.Conv1d(in_channels, out_channels, kernel_size=1, bias=False)
                         if in_channels != out_channels else nn.Identity())

    def forward(self, x):
        x_padded = F.pad(x, (self.causal_pad, 0))
        out = self.conv(x_padded)
        out = self.bn(out)
        out = self.relu(out)
        out = self.dropout(out)
        return out + self.res_proj(x)


class TCNModel(MultitaskHeadsMixin, nn.Module):
    def __init__(self, input_size, num_channels, kernel_size, n_buckets,
                 dropout=0.3, use_multitask=True):
        super().__init__()
        layers = []
        in_ch = input_size
        for i, out_ch in enumerate(num_channels):
            layers.append(CausalConvBlock(in_ch, out_ch, kernel_size, 2 ** i, dropout))
            in_ch = out_ch
        self.tcn = nn.Sequential(*layers)
        hidden_size = num_channels[-1]
        self.attn = TemporalAttention(hidden_size)
        self.norm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.fc_bucket = nn.Linear(hidden_size, n_buckets)
        self.fc_days = nn.Linear(hidden_size, 1)
        self.use_multitask = use_multitask
        if use_multitask:
            self._build_multitask_heads(hidden_size)

    def forward(self, x, lengths=None):
        out = self.tcn(x.permute(0, 2, 1)).permute(0, 2, 1)
        pooled = self.attn(out, lengths)
        return self._shared_head_forward(pooled)


class DropPath(nn.Module):
    """Stochastic Depth."""
    def __init__(self, p=0.1):
        super().__init__()
        self.p = p

    def forward(self, x):
        if not self.training or self.p == 0:
            return x
        keep = 1 - self.p
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        mask = torch.bernoulli(torch.full(shape, keep, device=x.device))
        return x * mask / keep


class ResidualGRUBlock(nn.Module):
    """Pre-LN -> GRU -> Dropout -> DropPath -> residual add."""
    def __init__(self, hidden, dropout=0.3, drop_path=0.1):
        super().__init__()
        self.ln = nn.LayerNorm(hidden)
        self.gru = nn.GRU(hidden, hidden, batch_first=True)
        self.dp = nn.Dropout(dropout)
        self.drop_path = DropPath(drop_path)

    def forward(self, x):
        h, _ = self.gru(self.ln(x))
        return x + self.drop_path(self.dp(h))


class DeepResidualGRU(MultitaskHeadsMixin, nn.Module):
    def __init__(self, input_size, hidden=128, n_layers=4, n_buckets=10,
                 dropout=0.3, drop_path=0.1, use_multitask=True):
        super().__init__()
        self.input_proj = nn.Linear(input_size, hidden)
        self.blocks = nn.ModuleList([
            ResidualGRUBlock(hidden, dropout, drop_path * (i + 1) / n_layers)
            for i in range(n_layers)
        ])
        self.attn = TemporalAttention(hidden)
        self.norm = nn.LayerNorm(hidden)
        self.dropout = nn.Dropout(dropout)
        self.fc_bucket = nn.Linear(hidden, n_buckets)
        self.fc_days = nn.Linear(hidden, 1)
        self.use_multitask = use_multitask
        if use_multitask:
            self._build_multitask_heads(hidden)

    def forward(self, x, lengths=None):
        h = self.input_proj(x)
        for blk in self.blocks:
            h = blk(h)
        pooled = self.attn(h, lengths)
        return self._shared_head_forward(pooled)


def create_model(model_type: str, input_size: int, config: dict) -> nn.Module:
    """Factory unificada — idéntica a la celda 28 del training."""
    n_buckets = config['n_time_buckets'] + 1
    use_multitask = config['use_multitask']

    if model_type in ('GRU', 'RNN', 'LSTM'):
        mc = config['model_configs'][model_type]
        cls = {'GRU': GRUModel, 'RNN': RNNModel, 'LSTM': LSTMModel}[model_type]
        return cls(input_size, mc['hidden_size'], mc['num_layers'],
                   n_buckets, mc['dropout'], use_multitask)

    if model_type == 'TCN':
        mc = config['model_configs']['TCN']
        return TCNModel(input_size, mc['num_channels'], mc['kernel_size'],
                        n_buckets, mc['dropout'], use_multitask)

    if model_type == 'DEEP_GRU':
        mc = config['model_configs']['DEEP_GRU']
        return DeepResidualGRU(input_size=input_size, hidden=mc['hidden_size'],
                               n_layers=mc['num_layers'], n_buckets=n_buckets,
                               dropout=mc['dropout'], drop_path=mc['drop_path'],
                               use_multitask=use_multitask)

    raise ValueError(f"Model '{model_type}' no implementado.")

print("Imports + arquitecturas OK")


Imports + arquitecturas OK


## 2. Cargar artefactos v1.9.0

Fuente única de verdad: `inference_bundle_v1_9_0.json` (celda 50 del training).
De ahí salen modelo elegido, decoder `q*`, temperaturas y política de abstención.


In [6]:
print(f"\n{'='*80}")
print("  INFERENCIA STANDALONE — be_payment_reminder — SeqNN v1.9.0")
print(f"{'='*80}")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
ARTIFACT_DIR = Path('artifacts_v1_9_0')
assert ARTIFACT_DIR.exists(), f"No existe {ARTIFACT_DIR}"

# ── 2.1 — inference_bundle: manifiesto maestro ──────────────────────────────
with open(ARTIFACT_DIR / 'inference_bundle_v1_9_0.json') as f:
    BUNDLE = json.load(f)

HMM_FEATURES         = BUNDLE['hmm_features']            # 47 features, lista canónica
BUCKET_EDGES         = BUNDLE['bucket_edges']            # [0,1,2,3,4,5,7,14,30]
BUCKET_CENTERS       = BUNDLE['bucket_centers']          # 10 centros
CLIP_ZSCORE          = BUNDLE['clip_zscore']
LOOKBACK             = BUNDLE['lookback_window']
TEMPERATURES         = BUNDLE['temperatures']            # dict {modelo: T}
BEST_MODEL_FINAL     = BUNDLE['best_model_type_final']   # 'RNN_CAL' | 'ENSEMBLE_CAL' | ...
OPTIMAL_Q            = BUNDLE['best_q_final']
ABSTENTION_POLICY    = BUNDLE['abstention_policy']       # 'logreg' | 'p_bucket0'
ABSTENTION_THRESHOLD = BUNDLE['abstention_threshold']

print(f"  Bundle: notebook {BUNDLE['notebook_version']}")
print(f"    Modelo final:  {BEST_MODEL_FINAL}")
print(f"    Decoder q*:    {OPTIMAL_Q}")
print(f"    Features:      {len(HMM_FEATURES)}")
print(f"    Buckets:       {len(BUCKET_CENTERS)}  edges={BUCKET_EDGES}")
print(f"    Abstención:    policy={ABSTENTION_POLICY}, thr={ABSTENTION_THRESHOLD:.3f}")

# ── 2.2 — CONFIG (necesario para create_model) ──────────────────────────────
# CONFIG vive dentro de cada .pt. Tomamos cualquiera para leer model_configs.
pt_files = sorted(ARTIFACT_DIR.glob('*_model_v1_9_0.pt'))
assert pt_files, "No hay archivos *_model_v1_9_0.pt en el artifact dir"
CONFIG = torch.load(pt_files[0], map_location='cpu')['config']

# ── 2.3 — decidir qué modelo(s) cargar ──────────────────────────────────────
IS_ENSEMBLE = BEST_MODEL_FINAL.upper().startswith('ENSEMBLE')
if IS_ENSEMBLE:
    MODELS_TO_LOAD = list(CONFIG['models_to_train'])     # ensemble = promedio de todos
else:
    MODELS_TO_LOAD = [BEST_MODEL_FINAL.replace('_CAL', '')]
print(f"\n  Modelos a cargar: {MODELS_TO_LOAD}")

# ── 2.4 — cargar pesos ──────────────────────────────────────────────────────
INPUT_SIZE = len(HMM_FEATURES)
models = {}
for mt in MODELS_TO_LOAD:
    ckpt = torch.load(ARTIFACT_DIR / f'{mt}_model_v1_9_0.pt', map_location=device)
    m = create_model(mt, INPUT_SIZE, CONFIG)
    m.load_state_dict(ckpt['model_state_dict'])           # falla ruidoso si hay mismatch
    models[mt] = m.to(device).eval()
    n_params = sum(p.numel() for p in m.parameters())
    print(f"    {mt:9s} cargado — {n_params:>9,} params  (T={TEMPERATURES.get(mt, 'n/a')})")
    del ckpt

# ── 2.5 — scaler ────────────────────────────────────────────────────────────
scaler = joblib.load(ARTIFACT_DIR / 'scaler_main.joblib')
assert scaler.n_features_in_ == INPUT_SIZE, (
    f"Scaler espera {scaler.n_features_in_} features, manifest tiene {INPUT_SIZE}")
print(f"\n  Scaler OK: {scaler.n_features_in_} features")

# ── 2.6 — clasificador de censura (para abstención opcional) ────────────────
CENSORING = None
_clf_path = ARTIFACT_DIR / 'censoring_classifier_v1_9_0.pkl'
if _clf_path.exists():
    with open(_clf_path, 'rb') as f:
        CENSORING = pickle.load(f)
    print(f"  Clasificador de censura OK (AUC val={CENSORING.get('auc_roc_val', float('nan')):.3f})")

gc.collect()
print(f"\n  device = {device}")



  INFERENCIA STANDALONE — be_payment_reminder — SeqNN v1.9.0
  Bundle: notebook v1.9.0
    Modelo final:  TCN_CAL
    Decoder q*:    0.15
    Features:      47
    Buckets:       10  edges=[0, 1, 2, 3, 4, 5, 7, 14, 30]
    Abstención:    policy=p_bucket0, thr=0.100

  Modelos a cargar: ['TCN']
    TCN       cargado —   387,727 params  (T=2.0070552197295317)

  Scaler OK: 47 features
  Clasificador de censura OK (AUC val=0.689)

  device = cuda


## 3. Funciones helper — feature engineering + inferencia

In [7]:
# ═══════════════════════════════════════════════════════════════════════════
#  3.1 — FEATURE ENGINEERING (réplica exacta de la celda 15 del training v1.9.0)
# ═══════════════════════════════════════════════════════════════════════════

def signed_log_expr(col_name, alias):
    """log con signo para montos que pueden ser negativos."""
    col = pl.col(col_name).fill_null(0).cast(pl.Float64)
    return (col.sign() * (col.abs() + 1).log()).alias(alias)


def build_hmm_features_v2(df: pl.DataFrame, config: dict) -> pl.DataFrame:
    """16 features baseline — idéntico a build_hmm_features_v2 del training."""
    return df.with_columns([
        pl.col('attempts_7d').fill_null(0).add(1).log().alias('log_attempts_7d'),
        pl.col('attempts_30d').fill_null(0).add(1).log().alias('log_attempts_30d'),
        pl.col('rejects_7d').fill_null(0).add(1).log().alias('log_rejects_7d'),
        (pl.col('attempts_7d').fill_null(0) + 1).log().alias('log_attempts_7d_new'),
        (pl.col('rejects_7d').fill_null(0) + 1).log().alias('log_rejects_7d_new'),
        (pl.col('rejects_7d').fill_null(0) /
         (pl.col('attempts_7d').fill_null(0) + 1)).alias('retry_intensity_7d_norm'),
        pl.col('decline_rate_30d').fill_null(0).clip(0, 1).alias('decline_rate_30d_clipped'),
        pl.when(pl.col('days_since_last_attempt').is_null())
          .then(config['cap_days_since']).otherwise(pl.col('days_since_last_attempt'))
          .clip(0, config['cap_days_since']).alias('days_since_last_attempt_hmm'),
        pl.when(pl.col('days_since_last_reject').is_null())
          .then(config['cap_days_since']).otherwise(pl.col('days_since_last_reject'))
          .clip(0, config['cap_days_since']).alias('days_since_last_reject_hmm'),
        pl.when(pl.col('days_since_last_approval').is_null())
          .then(pl.lit(1)).otherwise(pl.lit(0)).alias('missing_last_approval'),
        pl.when(pl.col('days_since_last_approval').is_null())
          .then(config['impute_days_since_approval']).otherwise(pl.col('days_since_last_approval'))
          .clip(0, config['cap_days_since']).alias('days_since_last_approval_hmm'),
        pl.col('days_to_predicted_due').is_null().cast(pl.Int8).alias('missing_predicted_due'),
        pl.when(pl.col('days_to_predicted_due').is_null())
          .then(config['impute_days_to_due']).otherwise(pl.col('days_to_predicted_due'))
          .clip(config['cap_days_to_due_min'], config['cap_days_to_due_max'])
          .alias('days_to_predicted_due_hmm'),
        pl.col('payment_interval_stddev').is_null().cast(pl.Int8).alias('missing_interval_std'),
        pl.when(pl.col('payment_interval_stddev').is_null())
          .then(config['impute_payment_interval_std']).otherwise(pl.col('payment_interval_stddev'))
          .alias('payment_interval_stddev_hmm'),
        (pl.col('net_flow_30d').fill_null(0) /
         pl.when(pl.col('outflow_30d').fill_null(0).abs() < 1.0).then(pl.lit(1.0))
           .otherwise(pl.col('outflow_30d').fill_null(0).abs())
        ).clip(-10, 10).alias('net_flow_30d_ratio'),
        pl.when(pl.col('inflow_30d').fill_null(0) < 0.01)
          .then(pl.lit(1)).otherwise(pl.lit(0)).alias('no_inflow_30d'),
        (pl.col('outflow_30d').fill_null(0).abs() /
         pl.when(pl.col('inflow_30d').fill_null(0) < 0.01).then(pl.lit(0.01))
           .otherwise(pl.col('inflow_30d').fill_null(0))
        ).clip(0, 10).alias('liquidity_pressure'),
    ])


def build_expanded_features(df: pl.DataFrame) -> pl.DataFrame:
    """31 features expandidas — convención genérica `target_*` (v1.9.0).

    Cambio vs v1.8.x: las columnas del ciclo del merchant pasaron de
    `nflx_txn_*` / `nflx_months_active` a `target_txn_*` / `target_months_active`.
    """
    exprs = [
        # === BLOCK_TARGET_CYCLE (9) ===
        (pl.col('target_txn_m1').fill_null(0) + 1).log().alias('log_target_txn_m1'),
        (pl.col('target_txn_m2').fill_null(0) + 1).log().alias('log_target_txn_m2'),
        (pl.col('target_txn_m3').fill_null(0) + 1).log().alias('log_target_txn_m3'),
        (pl.col('target_txn_total_3m').fill_null(0) + 1).log().alias('log_target_txn_total_3m'),
        pl.col('target_months_active').fill_null(0).clip(0, 24).cast(pl.Float64)
          .alias('target_months_active_clipped'),
        (pl.col('days_since_last_payment').fill_null(365) + 1).log()
          .alias('log_days_since_last_payment'),
        pl.col('days_since_last_payment').is_null().cast(pl.Int8)
          .alias('missing_days_since_payment'),
        (2 * np.pi * pl.col('billing_day_of_month').fill_null(15).cast(pl.Float64) / 31)
          .sin().alias('billing_day_sin'),
        (2 * np.pi * pl.col('billing_day_of_month').fill_null(15).cast(pl.Float64) / 31)
          .cos().alias('billing_day_cos'),
        # === BLOCK_LIQUIDITY_SHORT (9) ===
        (pl.col('inflow_7d').fill_null(0).abs() + 1).log().alias('log_inflow_7d'),
        (pl.col('outflow_7d').fill_null(0).abs() + 1).log().alias('log_outflow_7d'),
        signed_log_expr('net_flow_7d', 'log_signed_net_flow_7d'),
        (pl.col('inflow_14d').fill_null(0).abs() + 1).log().alias('log_inflow_14d'),
        (pl.col('outflow_14d').fill_null(0).abs() + 1).log().alias('log_outflow_14d'),
        signed_log_expr('net_flow_14d', 'log_signed_net_flow_14d'),
        (pl.col('inflow_30d').fill_null(0).abs() + 1).log().alias('log_inflow_30d'),
        (pl.col('outflow_30d').fill_null(0).abs() + 1).log().alias('log_outflow_30d'),
        signed_log_expr('net_flow_30d', 'log_signed_net_flow_30d'),
        # === BLOCK_RETRY_FINE (7) ===
        pl.col('attempts_in_day').fill_null(0).clip(0, 20).cast(pl.Float64)
          .alias('attempts_in_day_clipped'),
        pl.col('rejects_in_day').fill_null(0).clip(0, 20).cast(pl.Float64)
          .alias('rejects_in_day_clipped'),
        pl.col('approvals_in_day').fill_null(0).clip(0, 20).cast(pl.Float64)
          .alias('approvals_in_day_clipped'),
        (pl.col('attempts_14d').fill_null(0) + 1).log().alias('log_attempts_14d'),
        (pl.col('rejects_14d').fill_null(0) + 1).log().alias('log_rejects_14d'),
        (pl.col('approvals_14d').fill_null(0) + 1).log().alias('log_approvals_14d'),
        (pl.col('approvals_30d').fill_null(0) + 1).log().alias('log_approvals_30d'),
        # === BLOCK_AMOUNTS (4) ===
        (pl.col('amount_mxn').fill_null(0) + 1).log().alias('log_amount_mxn'),
        pl.col('amount_mxn').is_null().cast(pl.Int8).alias('missing_amount_mxn'),
        (pl.col('plan_amount').fill_null(0) + 1).log().alias('log_plan_amount'),
        pl.col('payment_interval_avg').fill_null(30).clip(0, 90).cast(pl.Float64)
          .alias('payment_interval_avg_clipped'),
        # === BLOCK_DEMOGRAPHIC_SAFE (2) ===
        pl.col('days_on_book').fill_null(0).clip(0, 5000).cast(pl.Float64)
          .alias('days_on_book_clipped'),
        pl.col('user_age_years').fill_null(35).clip(18, 90).cast(pl.Float64)
          .alias('user_age_years_clipped'),
    ]
    return df.with_columns(exprs)


# ═══════════════════════════════════════════════════════════════════════════
#  3.2 — INFERENCIA + DECODERS
# ═══════════════════════════════════════════════════════════════════════════

def _softmax(logits):
    shifted = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=1, keepdims=True)


@torch.no_grad()
def extract_logits(model, X, device, batch_size=8192):
    """Logits raw del bucket head. Sin lengths — coincide con el training,
    que entrena y evalúa llamando model(X) sin máscara de longitud."""
    model.eval()
    out = []
    for s in range(0, len(X), batch_size):
        xb = torch.from_numpy(X[s:s + batch_size]).float().to(device)
        logits_b, _, _ = model(xb)
        out.append(logits_b.cpu().numpy())
        del xb
    return np.concatenate(out, 0)


def calibrated_probs(logits, T):
    """Temperature scaling (Guo et al. 2017)."""
    return _softmax(logits / T)


def decode_quantile(probs, centers, q=0.35):
    """Decoder de cuantil sobre la CDF de buckets — idéntico al training."""
    centers = np.asarray(centers, dtype=np.float64)
    idx = np.argsort(centers)
    sc, sp = centers[idx], probs[:, idx]
    cp = np.cumsum(sp, axis=1)
    N, K = probs.shape
    out = np.zeros(N)
    for i in range(N):
        j = min(np.searchsorted(cp[i], q, side='left'), K - 1)
        if j == 0:
            out[i] = sc[0]
        else:
            lo, hi = cp[i, j - 1], cp[i, j]
            out[i] = sc[j - 1] + (q - lo) / (hi - lo) * (sc[j] - sc[j - 1]) \
                     if hi - lo > 1e-10 else sc[j]
    return out


def cumulative_attempt_probs(probs, bucket_edges):
    """P(intento <= h) para h in {3,7,14,30}, derivado de bucket_edges.

    Bucket k (1-indexado) cubre [edges[k-1], edges[k]); se incluye en la suma
    si su borde derecho edges[k] <= h. El bucket 0 (no-event) nunca se cuenta.
    Robusto a cambios futuros en bucket_edges.
    """
    edges = list(bucket_edges)

    def last_bucket_le(h):
        k = 0
        for i in range(1, len(edges)):
            if edges[i] <= h:
                k = i
        return k

    return {h: probs[:, 1:last_bucket_le(h) + 1].sum(axis=1) for h in (3, 7, 14, 30)}


def build_censoring_features(X_seq):
    """[ultimo_paso | promedio] — entrada del clasificador de censura (16g)."""
    return np.concatenate([X_seq[:, -1, :], X_seq.mean(axis=1)], axis=1)


print("Helpers definidos: feature engineering + decoders + abstención")


Helpers definidos: feature engineering + decoders + abstención


## 4. Cargar datos de inferencia desde BigQuery

> **Verificar `TABLE_EVENTS`.** El modelo v1.9.0 se entrenó con el merchant
> genérico (`recurrent_charger_google_one_merchant_exp_v5_0_1`). La tabla de
> inferencia debe tener el **mismo esquema de columnas** (`target_txn_*`,
> `attempts_*`, `inflow_*`, …). Si tu tabla usa prefijo `g1_*` o `nflx_*`, el
> bloque de rename de abajo lo normaliza automáticamente a `target_*`.


In [8]:
# ── CONFIG DE INFERENCIA — revisar antes de correr ──────────────────────────
PROJECT          = CONFIG['project']
DATASET          = CONFIG['dataset']
TABLE_EVENTS     = 'recurrent_charger_google_one_merchant_exp_v5_0_1'  # <-- AJUSTAR
EXPERIMENT_START = datetime(2026, 5, 25, tzinfo=timezone.utc)
EXPERIMENT_END   = datetime(2026, 6, 26, tzinfo=timezone.utc)
MIN_RECORDS      = 50_000

print(f"\n  Tabla:   {PROJECT}.{DATASET}.{TABLE_EVENTS}")
print(f"  Ventana: {EXPERIMENT_START.date()} -> {EXPERIMENT_END.date()}")
print(f"  Lookback: {LOOKBACK} eventos/usuario")

client = bigquery.Client(project=PROJECT)

print(f"\n  PASO 1 — Cargando historial (SELECT *)...")
t0 = time.time()
query = f"SELECT * FROM `{PROJECT}.{DATASET}.{TABLE_EVENTS}`"
df_hist = pl.from_arrow(client.query(query).to_arrow())
gc.collect()

# ── Normalizar prefijo del merchant a `target_*` ────────────────────────────
# El SQL v5.0.1 genérico ya expone target_txn_*; si la tabla viene de un
# merchant con prefijo legado (g1_*, nflx_*) lo renombramos aquí.
rename_map = {}
for c in df_hist.columns:
    if c.startswith('g1_'):
        rename_map[c] = 'target_' + c[3:]
    elif c.startswith('nflx_'):
        rename_map[c] = 'target_' + c[5:]
if rename_map:
    df_hist = df_hist.rename(rename_map)
    print(f"     Renombradas {len(rename_map)} columnas a convención target_*")

df_hist = df_hist.sort('userIdentifier', 'event_ts')
print(f"  OK — {df_hist.height:,} rows, "
      f"{df_hist['userIdentifier'].n_unique():,} usuarios ({time.time()-t0:.1f}s)")

# ── last_status + last_amount_mxn por usuario (vectorizado) ─────────────────
df_last_status = (
    df_hist.sort('userIdentifier', 'event_ts')
           .group_by('userIdentifier').last()
           .select(['userIdentifier', 'event_ts', 'status_type', 'amount_mxn'])
           .rename({'event_ts': 'last_event_ts', 'status_type': 'last_status',
                    'amount_mxn': 'last_amount_mxn'})
)
print(f"  last_status / last_amount_mxn: {df_last_status.height:,} usuarios")

# ── Quedarse con los últimos LOOKBACK eventos por usuario ───────────────────
df_hist = (
    df_hist.sort('userIdentifier', 'event_ts')
           .with_columns(pl.col('event_ts').rank('ordinal', descending=True)
                           .over('userIdentifier').alias('_rank'))
           .filter(pl.col('_rank') <= LOOKBACK)
           .drop('_rank')
           .sort('userIdentifier', 'event_ts')
)
print(f"  Reducido a últimos {LOOKBACK} eventos: {df_hist.height:,} rows")



  Tabla:   spin-aip-singularity-data-sb.recurrent_charger_netflix_merchant.recurrent_charger_google_one_merchant_exp_v5_0_1
  Ventana: 2026-05-25 -> 2026-06-26
  Lookback: 8 eventos/usuario

  PASO 1 — Cargando historial (SELECT *)...
  OK — 12,861,201 rows, 609,386 usuarios (144.0s)
  last_status / last_amount_mxn: 609,386 usuarios
  Reducido a últimos 8 eventos: 3,754,427 rows


## 5. Feature engineering + normalización con el scaler de training

In [9]:
print(f"\n  PASO 2 — Feature engineering...")
df_hist = build_hmm_features_v2(df_hist, CONFIG)
df_hist = build_expanded_features(df_hist)

missing = [f for f in HMM_FEATURES if f not in df_hist.columns]
if missing:
    raise ValueError(f"Features faltantes tras el FE: {missing}")
print(f"  OK — {len(HMM_FEATURES)} features construidas")

print(f"\n  PASO 3 — Normalizando con scaler de training...")
# HMM_FEATURES viene del manifest: ya es la lista canónica, única y ordenada
# exactamente como se fiteó el scaler. Se selecciona directo, sin dedup.
X_raw  = df_hist.select(HMM_FEATURES).fill_null(0).to_numpy().astype(np.float32)
X_norm = np.clip(scaler.transform(X_raw), -CLIP_ZSCORE, CLIP_ZSCORE)
del X_raw
gc.collect()
print(f"  OK — X_norm {X_norm.shape}")



  PASO 2 — Feature engineering...
  OK — 47 features construidas

  PASO 3 — Normalizando con scaler de training...
  OK — X_norm (3754427, 47)


## 6. Construcción de secuencias

Una secuencia por usuario, anclada en su evento más reciente. Equivale a la
ventana deslizante de `build_sequences` evaluada en el último evento:
se toman los últimos `LOOKBACK` pasos y se hace **left-padding** con ceros
(padding al frente, datos reales al final) — igual que en el training.


In [10]:
print(f"\n  PASO 4 — Construyendo secuencias ({LOOKBACK} timesteps)...")

users         = df_hist['userIdentifier'].to_list()
event_ts_list = df_hist['event_ts'].to_list()
n_features    = len(HMM_FEATURES)

# Agrupar índices por usuario preservando el orden temporal
user_indices = {}
for idx, u in enumerate(users):
    user_indices.setdefault(u, []).append(idx)

sequences, meta_users, meta_event_ts = [], [], []
for uid, indices in user_indices.items():
    indices = sorted(indices)
    seq = X_norm[indices[-LOOKBACK:]]                 # (<=LOOKBACK, n_features)
    if len(seq) < LOOKBACK:                           # left-pad con ceros
        pad = np.zeros((LOOKBACK - len(seq), n_features), dtype=np.float32)
        seq = np.vstack([pad, seq])
    sequences.append(seq)
    meta_users.append(uid)
    meta_event_ts.append(event_ts_list[indices[-1]])  # ancla = último evento

X_seq = np.stack(sequences).astype(np.float32)        # (N_users, LOOKBACK, n_features)
print(f"  OK — X_seq {X_seq.shape}")

del X_norm, df_hist, user_indices, sequences
gc.collect()



  PASO 4 — Construyendo secuencias (8 timesteps)...
  OK — X_seq (609386, 8, 47)


0

## 7. Inferencia + (opcional) abstención

Para cada modelo: logits → temperature scaling con su `T` → probs calibradas.
Si el bundle eligió `ENSEMBLE_CAL`, se promedian las probs calibradas de todos
los modelos (igual que la celda 16f del training).

`USE_ABSTENTION` viene **apagado** por defecto para preservar el comportamiento
de este notebook. El training v1.9.0 documenta explícitamente que la abstención
es un *componente débil conocido* (AUC bajo en test); enciéndelo solo si lo vas
a validar.


In [11]:
USE_ABSTENTION = False   # v1.9.0 marca la abstención como componente débil

print(f"\n  PASO 5 — Inferencia ({BEST_MODEL_FINAL}) en {device}...")

# Logits -> probs calibradas (y no calibradas, por si la política es p_bucket0)
cal_by_model, uncal_by_model = {}, {}
for mt, model in models.items():
    logits = extract_logits(model, X_seq, device)
    cal_by_model[mt]   = calibrated_probs(logits, TEMPERATURES[mt])
    uncal_by_model[mt] = _softmax(logits)

if IS_ENSEMBLE:
    cal_probs   = np.mean(list(cal_by_model.values()), axis=0)
    uncal_probs = np.mean(list(uncal_by_model.values()), axis=0)
else:
    mt0 = MODELS_TO_LOAD[0]
    cal_probs, uncal_probs = cal_by_model[mt0], uncal_by_model[mt0]

# Decoder de cuantil + probabilidades acumuladas
pred_days_arr = decode_quantile(cal_probs, BUCKET_CENTERS, q=OPTIMAL_Q)
cum = cumulative_attempt_probs(cal_probs, BUCKET_EDGES)
p_attempt_3d, p_attempt_7d = cum[3], cum[7]
p_attempt_14d, p_attempt_30d = cum[14], cum[30]
confidence = cal_probs[:, 1:].max(axis=1)

print(f"  OK — {len(pred_days_arr):,} predicciones")
print(f"     pred_days: median={np.median(pred_days_arr):.1f}d  "
      f"p90={np.percentile(pred_days_arr, 90):.1f}d")

# ── Abstención (opcional) ───────────────────────────────────────────────────
if USE_ABSTENTION:
    if ABSTENTION_POLICY == 'logreg' and CENSORING is not None:
        feats_s  = CENSORING['scaler'].transform(build_censoring_features(X_seq))
        p_censor = 1.0 - CENSORING['clf'].predict_proba(feats_s)[:, 1]
        thr      = CENSORING['threshold']
    else:  # 'p_bucket0' — usa P(bucket 0) sin calibrar del modelo/ensemble
        p_censor = uncal_probs[:, 0]
        thr      = ABSTENTION_THRESHOLD
    abstain_mask = p_censor > thr
    print(f"  Abstención ({ABSTENTION_POLICY}, thr={thr:.3f}): "
          f"{abstain_mask.sum():,} usuarios ({abstain_mask.mean()*100:.1f}%)")
else:
    p_censor     = np.zeros(len(cal_probs))
    abstain_mask = np.zeros(len(cal_probs), dtype=bool)
    print(f"  Abstención: DESACTIVADA")

del X_seq
gc.collect()



  PASO 5 — Inferencia (TCN_CAL) en cuda...
  OK — 609,386 predicciones
     pred_days: median=2.0d  p90=8.2d
  Abstención: DESACTIVADA


0

## 8. Construcción de la scoring card `be_payment_reminder`

In [12]:
print(f"\n  PASO 6 — Construyendo scoring card...")

def to_dt(v):
    if isinstance(v, datetime):
        return v if v.tzinfo else v.replace(tzinfo=timezone.utc)
    if isinstance(v, str):
        return datetime.fromisoformat(v.replace('Z', '+00:00'))
    return None

records = []
for i in range(len(meta_users)):
    ets_dt    = to_dt(meta_event_ts[i])
    pred_d    = float(pred_days_arr[i])
    pred_date = (ets_dt + timedelta(days=pred_d)) if ets_dt else None
    conf      = float(confidence[i])
    p7, p14   = float(p_attempt_7d[i]), float(p_attempt_14d[i])

    if   pred_d <= 3:  timing = 'urgent'
    elif pred_d <= 7:  timing = 'short_term'
    elif pred_d <= 14: timing = 'medium_term'
    else:              timing = 'long_term'

    send_ts = (pred_date - timedelta(days=1)) if pred_date else None

    # Regla de acción — abstención manda primero (si está activada)
    if abstain_mask[i]:
        action = 'no_action'
    elif p7 >= 0.5 or (timing == 'urgent' and conf >= 0.2):
        action = 'send_reminder'
    elif p14 >= 0.4:
        action = 'monitor'
    else:
        action = 'no_action'

    codes = []
    if abstain_mask[i]:  codes.append('ABSTAIN_CENSORING')
    if timing == 'urgent': codes.append('URGENT_TIMING')
    if p7 >= 0.5:  codes.append('HIGH_P7D')
    if p14 >= 0.7: codes.append('HIGH_P14D')
    if conf >= 0.4:   codes.append('HIGH_CONFIDENCE')
    elif conf < 0.2:  codes.append('LOW_CONFIDENCE')
    if not codes: codes.append('STANDARD')

    records.append({
        'userIdentifier': meta_users[i],
        'last_attempt_ts': str(meta_event_ts[i]) if meta_event_ts[i] else None,
        'pred_next_attempt_date': pred_date.strftime('%Y-%m-%d') if pred_date else None,
        'pred_days_expected': round(pred_d, 2),
        'p_attempt_3d': round(float(p_attempt_3d[i]), 4),
        'p_attempt_7d': round(p7, 4),
        'p_attempt_14d': round(p14, 4),
        'p_attempt_30d': round(float(p_attempt_30d[i]), 4),
        'p_censor': round(float(p_censor[i]), 4),
        'confidence': round(conf, 4),
        'recommended_timing': timing,
        'recommended_send_ts': send_ts.strftime('%Y-%m-%d %H:%M:%S') if send_ts else None,
        'action': action,
        'reason_codes': '|'.join(codes),
    })

df_scoring = pl.DataFrame(records)

# Join last_status / last_amount_mxn
df_scoring = df_scoring.join(
    df_last_status.select(['userIdentifier', 'last_status', 'last_amount_mxn']),
    on='userIdentifier', how='left')

# Regla de negocio: si el último status es 'Approved', el siguiente cobro se
# ancla a último_intento + 60d (no se usa la predicción del modelo).
print(f"  Aplicando regla de negocio (last_status == 'Approved')...")
df_scoring = df_scoring.with_columns(
    pl.col('last_attempt_ts').str.to_datetime(format='%Y-%m-%d %H:%M:%S%Z', strict=False)
      .alias('_last_dt'),
    pl.col('pred_next_attempt_date').str.to_datetime(format='%Y-%m-%d', strict=False)
      .alias('_pred_dt'),
).with_columns(
    pl.when(pl.col('last_status').str.to_lowercase() == 'approved')
      .then(pl.col('_last_dt') + pl.duration(days=60))
      .otherwise(pl.col('_pred_dt'))
      .dt.strftime('%Y-%m-%d')
      .alias('pred_next_attempt_date')
).drop(['_last_dt', '_pred_dt'])

# Una fila por usuario (evento más reciente)
be_payment_reminder = (
    df_scoring.sort(['userIdentifier', 'last_attempt_ts'], descending=[False, True])
              .unique(subset=['userIdentifier'], keep='first')
)

# Filtro de ventana experimental
fecha_inicio_filtro = EXPERIMENT_START.strftime('%Y-%m-%d')
fecha_fin_filtro    = EXPERIMENT_END.strftime('%Y-%m-%d')
print(f"  Filtro de ventana: {fecha_inicio_filtro} a {fecha_fin_filtro}...")
be_payment_reminder = (
    be_payment_reminder
    .filter((pl.col('pred_next_attempt_date') >= fecha_inicio_filtro) &
            (pl.col('pred_next_attempt_date') <= fecha_fin_filtro))
    .sort('pred_next_attempt_date')
)
print(f"  OK — {len(be_payment_reminder):,} filas en ventana")



  PASO 6 — Construyendo scoring card...
  Aplicando regla de negocio (last_status == 'Approved')...
  Filtro de ventana: 2026-05-25 a 2026-06-26...
  OK — 66,614 filas en ventana


## 9. Estadísticas + guardado

In [13]:
print(f"\n  {'-'*60}")
print(f"  RESUMEN be_payment_reminder")
print(f"  {'-'*60}")
print(f"  Filas totales:      {len(be_payment_reminder):,}")
print(f"  Usuarios únicos:    {be_payment_reminder['userIdentifier'].n_unique():,}")
print(f"  Rango pred_date:    {be_payment_reminder['pred_next_attempt_date'].min()} -> "
      f"{be_payment_reminder['pred_next_attempt_date'].max()}")
print(f"  pred_days_expected: median={be_payment_reminder['pred_days_expected'].median():.1f}d  "
      f"p90={be_payment_reminder['pred_days_expected'].quantile(0.90):.1f}d")
print(f"  confidence:         median={be_payment_reminder['confidence'].median():.3f}  "
      f"p10={be_payment_reminder['confidence'].quantile(0.10):.3f}")

print(f"\n  Distribución de action:")
for row in (be_payment_reminder.group_by('action').agg(pl.len().alias('n'))
            .sort('n', descending=True).iter_rows(named=True)):
    print(f"    {row['action']:20s}  {row['n']:>7,}  "
          f"({row['n']/len(be_payment_reminder)*100:.1f}%)")

print(f"\n  Distribución de timing:")
for row in (be_payment_reminder.group_by('recommended_timing').agg(pl.len().alias('n'))
            .sort('n', descending=True).iter_rows(named=True)):
    print(f"    {row['recommended_timing']:20s}  {row['n']:>7,}  "
          f"({row['n']/len(be_payment_reminder)*100:.1f}%)")

# Gate 50K
n_total = be_payment_reminder['userIdentifier'].n_unique()
if n_total >= MIN_RECORDS:
    print(f"\n  GATE {MIN_RECORDS:,} PASS: {n_total:,} usuarios")
else:
    print(f"\n  GATE {MIN_RECORDS:,} FAIL: {n_total:,} / {MIN_RECORDS:,}")

# Guardar
out_parquet = ARTIFACT_DIR / 'be_payment_reminder.parquet'
out_csv     = ARTIFACT_DIR / 'be_payment_reminder.csv'
be_payment_reminder.write_parquet(out_parquet)
be_payment_reminder.write_csv(out_csv)
print(f"\n  Guardado: {out_parquet}")
print(f"  Guardado: {out_csv}")

print(f"\n  Preview (5 filas):")
with pl.Config(tbl_cols=-1, tbl_width_chars=220):
    print(be_payment_reminder.head(5))



  ------------------------------------------------------------
  RESUMEN be_payment_reminder
  ------------------------------------------------------------
  Filas totales:      66,614
  Usuarios únicos:    66,614
  Rango pred_date:    2026-05-25 -> 2026-06-26
  pred_days_expected: median=6.0d  p90=8.9d
  confidence:         median=0.471  p10=0.377

  Distribución de action:
    no_action              61,377  (92.1%)
    send_reminder           3,900  (5.9%)
    monitor                 1,337  (2.0%)

  Distribución de timing:
    short_term             37,075  (55.7%)
    medium_term            25,647  (38.5%)
    urgent                  3,892  (5.8%)

  GATE 50,000 PASS: 66,614 usuarios

  Guardado: artifacts_v1_9_0/be_payment_reminder.parquet
  Guardado: artifacts_v1_9_0/be_payment_reminder.csv

  Preview (5 filas):
shape: (5, 16)
┌─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬──────────┬────────────┬─────────────┬───

## 10. Conteo de cobros predichos por día / semana / mes

Prerrequisito: haber ejecutado la celda de scoring (`be_payment_reminder` en
memoria).


In [14]:
print(f"\n{'='*80}")
print("  CONTEO DE COBROS PREDICHOS — Distribución Temporal")
print(f"{'='*80}")

assert 'be_payment_reminder' in dir(), "Ejecutar la celda de scoring primero"
assert len(be_payment_reminder) > 0, "be_payment_reminder está vacío"

df_counts = (
    be_payment_reminder
    .filter(pl.col('pred_next_attempt_date').is_not_null())
    .with_columns(pl.col('pred_next_attempt_date').str.to_date('%Y-%m-%d').alias('pred_date'))
)

# ── 1. Conteo diario ────────────────────────────────────────────────────────
daily_counts = (
    df_counts.group_by('pred_date')
    .agg([pl.col('userIdentifier').n_unique().alias('usuarios_unicos'),
          pl.len().alias('total_rows')])
    .sort('pred_date')
    .with_columns(pl.col('usuarios_unicos').cum_sum().alias('usuarios_acumulados'),
                  pl.col('pred_date').dt.strftime('%A').alias('dia_semana'))
)
print(f"\n  {'-'*70}")
print(f"  CONTEO DIARIO")
print(f"  {'-'*70}")
print(f"  {'Fecha':<14s} {'Día':<12s} {'Usuarios':>10s} {'Acumulado':>12s}")
for row in daily_counts.iter_rows(named=True):
    print(f"  {str(row['pred_date']):<14s} {row['dia_semana']:<12s} "
          f"{row['usuarios_unicos']:>10,} {row['usuarios_acumulados']:>12,}")
print(f"\n  Total usuarios únicos en ventana: {df_counts['userIdentifier'].n_unique():,}")

# ── 2. Conteo mensual ───────────────────────────────────────────────────────
monthly_counts = (
    df_counts.with_columns(pl.col('pred_date').dt.strftime('%Y-%m').alias('mes'))
    .group_by('mes')
    .agg([pl.col('userIdentifier').n_unique().alias('usuarios_unicos'),
          pl.col('pred_date').min().alias('primer_dia'),
          pl.col('pred_date').max().alias('ultimo_dia')])
    .sort('mes')
)
print(f"\n  {'-'*70}")
print(f"  CONTEO MENSUAL")
print(f"  {'-'*70}")
print(f"  {'Mes':<10s} {'Rango':<27s} {'Usuarios':>10s}")
for row in monthly_counts.iter_rows(named=True):
    rango = f"{row['primer_dia']} -> {row['ultimo_dia']}"
    print(f"  {row['mes']:<10s} {rango:<27s} {row['usuarios_unicos']:>10,}")

# ── 3. Resumen rápido ───────────────────────────────────────────────────────
avg_daily = daily_counts['usuarios_unicos'].mean()
max_daily = daily_counts['usuarios_unicos'].max()
max_day   = daily_counts.filter(pl.col('usuarios_unicos') == max_daily)['pred_date'][0]
print(f"\n  {'-'*70}")
print(f"  RESUMEN RÁPIDO")
print(f"  {'-'*70}")
print(f"  Usuarios únicos totales:   {df_counts['userIdentifier'].n_unique():,}")
print(f"  Promedio diario:           {avg_daily:,.0f} usuarios/día")
print(f"  Día pico:                  {max_day} ({max_daily:,} usuarios)")
print(f"  Días con cobros predichos: {daily_counts.height}")

n_total = df_counts['userIdentifier'].n_unique()
if n_total >= 50_000:
    print(f"\n  GATE 50K PASS: {n_total:,} usuarios únicos")
else:
    print(f"\n  GATE 50K FAIL: {n_total:,} / 50,000 — faltan {50_000 - n_total:,}")

# ── 4. Conteo semanal ───────────────────────────────────────────────────────
weekly_counts = (
    df_counts.with_columns(pl.col('pred_date').dt.strftime('%Y-W%W').alias('semana'))
    .group_by('semana')
    .agg(pl.col('userIdentifier').n_unique().alias('usuarios_unicos'))
    .sort('semana')
)
print(f"\n  {'-'*40}")
print(f"  CONTEO SEMANAL")
print(f"  {'-'*40}")
print(f"  {'Semana':<12s} {'Usuarios':>10s}")
for row in weekly_counts.iter_rows(named=True):
    print(f"  {row['semana']:<12s} {row['usuarios_unicos']:>10,}")



  CONTEO DE COBROS PREDICHOS — Distribución Temporal

  ----------------------------------------------------------------------
  CONTEO DIARIO
  ----------------------------------------------------------------------
  Fecha          Día            Usuarios    Acumulado
  2026-05-25     Monday            1,039        1,039
  2026-05-26     Tuesday             695        1,734
  2026-05-27     Wednesday           481        2,215
  2026-05-28     Thursday            344        2,559
  2026-05-29     Friday              688        3,247
  2026-05-30     Saturday            777        4,024
  2026-05-31     Sunday              836        4,860
  2026-06-01     Monday              746        5,606
  2026-06-02     Tuesday             780        6,386
  2026-06-03     Wednesday           753        7,139
  2026-06-04     Thursday            698        7,837
  2026-06-05     Friday              725        8,562
  2026-06-06     Saturday            709        9,271
  2026-06-07     Sunday    